# 3-gNB topology — mixed slice load

One gNB (gNB-0) hosts **1 eMBB + 2 URLLC** UEs.  
gNB-1 and gNB-2 are empty.  
We plot the eMBB and URLLC load at every gNB over time.

In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-chech')
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scenario_creator import create_multignb_env

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Ready')

## Configuration

In [ ]:
SLICE_TYPES = ('eMBB', 'URLLC', 'mMTC')

# step_dt = 1 ms matches slot_length so capacity ≈ 85 Mbps per gNB
TICK_S       = 0.001
SIMULATION_S = 10.0
N_TICKS      = int(SIMULATION_S / TICK_S)   # 10 000 ticks

# Standard triangle topology
GNB_CONFIGS = [
    {'id': 0, 'x':   0.0, 'y':   0.0, 'coverage_radius': 520.0, 'carrier_id': 0, 'n_prbs': 100},
    {'id': 1, 'x': 450.0, 'y':   0.0, 'coverage_radius': 520.0, 'carrier_id': 0, 'n_prbs': 100},
    {'id': 2, 'x': 225.0, 'y': 390.0, 'coverage_radius': 520.0, 'carrier_id': 0, 'n_prbs': 100},
]

print(f'Simulation: {SIMULATION_S:.0f} s  ({N_TICKS} ticks at {TICK_S*1000:.0f} ms/tick)')

## UE placement

All three UEs are placed near gNB-0 so they attach there automatically.  
`packet_size = bit_rate × step_dt` → exactly 1 packet per tick, fully deterministic load.

| UE | Slice | Bit rate | Packet size | PRBs / tick | Position |
|---|---|---|---|---|---|
| 0 | eMBB  | 12 Mbps | 12 000 bits | 15 | (30, 0)   |
| 1 | URLLC |  4 Mbps |  4 000 bits |  6 | (20, 20)  |
| 2 | URLLC |  4 Mbps |  4 000 bits |  6 | (20, −20) |

Expected steady-state at **gNB-0**: eMBB ≈ 0.15 (15/100 PRBs), URLLC ≈ 0.12 (12/100 PRBs).  
gNB-1 and gNB-2 have no UEs → load = 0.

In [ ]:
env = create_multignb_env(
    rng=np.random.default_rng(0),
    n=4,
    slots_per_step=1,
    L1_level=False,
    gnb_configs=GNB_CONFIGS,
    step_dt=TICK_S,
    mobility_dt=TICK_S,
    radio_substeps=1,
    max_episode_steps=N_TICKS + 1,
    max_prbs_per_ue=None,
)

# packet_size = bit_rate × step_dt  →  exactly 1 packet per tick, deterministic load
UE_SPECS = [
    dict(x=30.0, y=  0.0, slice_type='eMBB',  bit_rate=12e6, packet_size=12_000),
    dict(x=20.0, y= 20.0, slice_type='URLLC', bit_rate= 4e6, packet_size= 4_000),
    dict(x=20.0, y=-20.0, slice_type='URLLC', bit_rate= 4e6, packet_size= 4_000),
]

for spec in UE_SPECS:
    env.add_ue(
        x=spec['x'], y=spec['y'], vx=0., vy=0.,
        slice_type=spec['slice_type'],
        bit_rate=spec['bit_rate'],
        packet_size_bits=float(spec['packet_size']),
        traffic_model='fixed_packet_cbr',
    )

print('UEs created:')
for u in env.get_all_ues():
    print(f'  UE-{u.id}  ({u.x:5.0f}, {u.y:4.0f})  slice={u.slice_type:<6}  gNB={u.serving_gnb}  connected={u.connected}')

## Run simulation and record load

In [ ]:
records = []

# Sample every 10 ticks (= 10 ms) to keep the DataFrame manageable
SAMPLE_EVERY = 10

for tick in range(N_TICKS):
    env.step(0)

    if tick % SAMPLE_EVERY == 0:
        loads = env.get_slice_loads()   # dict[(gnb_id, slice_type)] → float
        row = {'time_s': (tick + 1) * TICK_S}
        for gnb in range(3):
            for sl in SLICE_TYPES:
                row[f'gNB{gnb}_{sl}'] = float(loads.get((gnb, sl), 0.0))
        records.append(row)

df = pd.DataFrame(records)
env.close()

print(f'Collected {len(df)} samples')
print(df.tail(3).to_string(index=False))

## Load per gNB per slice

In [ ]:
gnb_colors  = ['steelblue', 'tomato', 'seagreen']
slice_plot  = ['eMBB', 'URLLC']   # mMTC omitted (no UEs)
slice_title = {'eMBB': 'eMBB (1 UE @ 12 Mbps)', 'URLLC': 'URLLC (2 UEs @ 4 Mbps each)'}

fig, axes = plt.subplots(len(slice_plot), 1, figsize=(12, 5), sharex=True)

for ax, sl in zip(axes, slice_plot):
    for gnb, color in enumerate(gnb_colors):
        col = f'gNB{gnb}_{sl}'
        ax.plot(df['time_s'], df[col], color=color, lw=1.8, label=f'gNB-{gnb}')
    ax.set_ylabel('load  (PRBs used / total PRBs)')
    ax.set_ylim(-0.05, 1.10)
    ax.set_title(slice_title[sl])
    ax.legend(loc='right')
    ax.grid(axis='y', alpha=0.4)

axes[-1].set_xlabel('time (s)')
plt.suptitle('3-gNB topology — load per gNB per slice\n'
             'gNB-0 hosts 1 eMBB + 2 URLLC UEs; gNB-1 and gNB-2 are empty',
             y=1.01)
plt.tight_layout()
plt.show()

# Print steady-state averages
print('\nSteady-state average load (last 5 s):')
tail = df[df['time_s'] >= SIMULATION_S / 2]
for gnb in range(3):
    for sl in SLICE_TYPES:
        mean = tail[f'gNB{gnb}_{sl}'].mean()
        if mean > 0.01:
            print(f'  gNB-{gnb}  {sl:<6}  load = {mean:.3f}')